# Hiligaynon NER: Deep Error Diagnostics & Confusion Matrix

This notebook generates granular confusion matrices to expose semantic tagging boundaries. It is structured to help manually inspect and isolate misclassifications, heavily centering on the `ORG` vs `GPE` constraint overlap and the linguistic challenges introduced by Taglish (code-switching).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os
import itertools

print('Diagnostic libraries loaded.')

### 1. Generating the Heatmap

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title='NER Confusion Matrix', cmap='Reds'):
    """
    Plots a seaborn heatmap of the classification confusion matrix.
    """
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, 
                xticklabels=classes, yticklabels=classes, 
                cbar=False, linewidths=.5)
    
    plt.ylabel('Actual True Tags', fontsize=12, fontweight='bold')
    plt.xlabel('Predicted Tags', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Example usage (replace with actual flattened arrays from true labels and predictions):
sample_classes = ['B-ORG', 'I-ORG', 'B-GPE', 'I-GPE', 'B-PER', 'O']
# plot_confusion_matrix(y_true_flat, y_pred_flat, classes=sample_classes)

### 2. Hunting Down Misclassifications (ORG vs GPE & Taglish)
The function below is designed to filter out exact sentences where a specific true tag is systematically predicted as a designated false tag.

In [ ]:
def inspect_errors(sentences, true_labels, pred_labels, true_target='B-ORG', pred_target='B-GPE'):
    """
    Extracts contextual sentences where models confused true_target for pred_target.
    """
    error_instances = []
    
    for sentence_idx, (words, t_tags, p_tags) in enumerate(zip(sentences, true_labels, pred_labels)):
        for word_idx, (word, t_tag, p_tag) in enumerate(zip(words, t_tags, p_tags)):
            if t_tag == true_target and p_tag == pred_target:
                error_instances.append({
                    'sentence': " ".join(words),
                    'fault_word': word,
                    'context': f"True: {t_tag} | Pred: {p_tag}"
                })
                
    return error_instances

# TODO: Pass outputs from eval_script.py into this function
# org_gpe_errors = inspect_errors(sentences, true_labels, pred_labels, 'B-ORG', 'B-GPE')
# for err in org_gpe_errors[:5]:
#     print(f"Word: '{err['fault_word']}' -> {err['context']}\nSentence: {err['sentence']}\n")